In [0]:
schema = "workspace.bronze"
source_details_df = spark.table("workspace.control.source_file_details").filter("is_active = true")
source_specs_df = spark.table("workspace.control.source_file_specifications").filter("is_active = true")

In [0]:
for row in source_details_df.collect():
    source_id = row['source_id']
    file_path = row['file_path']
    file_format = row['file_format']
    file_delimitor = row['file_delimitor']
    is_header = row['is_header']
    file_name = row['file_name']
    load_type = row['load_type']
    table_name = file_name.replace("-", "_").replace(".", "_")
    
    spec_fields = (
        source_specs_df
        .filter(f"source_id = '{source_id}'")
        .orderBy("field_order")
        .select("field_name", "data_type")
        .collect()
    )
    
    field_names = [f['field_name'] for f in spec_fields]
    field_types = [f['data_type'] for f in spec_fields]
    schema_str = ", ".join([f"{n} {t}" for n, t in zip(field_names, field_types)])
    
    read_options = {
        "delimiter": file_delimitor,
        "inferSchema": "false",
        "header": "true" if is_header else "false"
    }
    df = spark.read.format(file_format).options(**read_options).load(file_path)
    
    if is_header:
        df_cols = [c.lower() for c in df.columns]
        spec_cols = [c.lower() for c in field_names]
        if df_cols != spec_cols:
            raise Exception(f"Schema mismatch for {file_name}: file columns {df.columns} != spec {field_names}")
        df = df.select(field_names)
    else:
        df = df.toDF(*field_names)
    
    for name, dtype in zip(field_names, field_types):
        df = df.withColumn(name, df[name].cast(dtype))
    
    write_mode = "overwrite" if load_type.lower() == "full_load" else "append"
    df.write.format("delta").mode(write_mode).saveAsTable(f"{schema}.{table_name}")